In [1]:
import os
import re
import math
import json
import hashlib
import unicodedata
from datetime import datetime

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import types as T

# ============================================================
# TENTATIVA DE CAMADA SEMANTICA LOCAL
# ============================================================

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_OK = True
except Exception:
    SKLEARN_OK = False

from difflib import SequenceMatcher


# ============================================================
# VALIDACAO DE ENTRADA
# ============================================================

if spark.sql("show tables in gold like 'sinp_fat_ocorrencia_livro'").count() == 0:
    raise Exception("Tabela gold.sinp_fat_ocorrencia_livro não encontrada. Gere a fato do livro antes da classificação semântica.")


# ============================================================
# LIMPEZA DEFENSIVA
# ============================================================

tabelas_drop = [
    "tmp_ocorrencia_livro_texto_base",
    "tmp_ocorrencia_livro_texto_novo",
    "tmp_dim_classificacao_ocorrencia_livro_novo",
    "tmp_dim_classificacao_ocorrencia_livro_final",
    "sinp_dim_classificacao_ocorrencia_livro",
    "sinp_fat_ocorrencia_livro_classificada"
]

for t in tabelas_drop:
    spark.sql(f"drop table if exists gold.{t}")
    os.system(f"hdfs dfs -rm -r -skipTrash {path}{t} >/dev/null 2>&1")

spark.catalog.clearCache()
spark.sql("refresh table gold.sinp_fat_ocorrencia_livro")


# ============================================================
# TAXONOMIA FECHADA
# ============================================================

TAXONOMIA = [
    {
        "macroclasse": "SEGURANCA",
        "classe": "FUGA_EVASAO",
        "subclasse": "FUGA_CONSUMADA",
        "grau_risco": 10,
        "criticidade": "CRITICA",
        "patterns": [
            r"\bfuga\b", r"\bevas[aã]o\b", r"\bevadiu\b", r"\bforagid", r"\bempreendeu fuga\b"
        ],
        "examples": [
            "fuga consumada de interno",
            "evasao da unidade prisional",
            "interno evadiu"
        ]
    },
    {
        "macroclasse": "SEGURANCA",
        "classe": "FUGA_EVASAO",
        "subclasse": "TENTATIVA_DE_FUGA",
        "grau_risco": 9,
        "criticidade": "CRITICA",
        "patterns": [
            r"tentativa de fuga", r"tentou fugir", r"tentativa de evas[aã]o",
            r"serra", r"buraco na cela", r"rompimento de grade"
        ],
        "examples": [
            "tentativa de fuga",
            "interno tentou fugir",
            "tentativa de evasao da cela"
        ]
    },
    {
        "macroclasse": "SEGURANCA",
        "classe": "APREENSAO_ILICITO",
        "subclasse": "CELULAR_ELETRONICO",
        "grau_risco": 8,
        "criticidade": "ALTA",
        "patterns": [
            r"\bcelular\b", r"\btelefone\b", r"\bsmartphone\b", r"\bchip\b",
            r"\bcarregador\b", r"\bfone\b", r"\br[aá]dio\b", r"\baparelho eletr[oô]nico\b"
        ],
        "examples": [
            "apreensao de celular",
            "encontrado telefone na cela",
            "chip e carregador apreendidos"
        ]
    },
    {
        "macroclasse": "SEGURANCA",
        "classe": "APREENSAO_ILICITO",
        "subclasse": "DROGA_ENTORPECENTE",
        "grau_risco": 9,
        "criticidade": "CRITICA",
        "patterns": [
            r"\bdroga\b", r"\bentorpec", r"\bmaconha\b", r"\bcoca[ií]na\b",
            r"\bcrack\b", r"\bsubst[aâ]ncia\b", r"\btr[aá]fico\b"
        ],
        "examples": [
            "apreensao de droga",
            "entorpecente encontrado",
            "maconha apreendida"
        ]
    },
    {
        "macroclasse": "SEGURANCA",
        "classe": "APREENSAO_ILICITO",
        "subclasse": "ARMA_OBJETO_PERFUROCORTANTE",
        "grau_risco": 9,
        "criticidade": "CRITICA",
        "patterns": [
            r"\barma\b", r"\bfaca\b", r"\bl[aâ]mina\b", r"\bestoque\b",
            r"\bchucho\b", r"\bperfurocortante\b", r"\bobjeto cortante\b"
        ],
        "examples": [
            "arma artesanal apreendida",
            "faca encontrada na cela",
            "objeto perfurocortante"
        ]
    },
    {
        "macroclasse": "SEGURANCA",
        "classe": "CONTRABANDO",
        "subclasse": "ARREMESSO_OU_INGRESSO_ILICITO",
        "grau_risco": 8,
        "criticidade": "ALTA",
        "patterns": [
            r"\barremesso\b", r"\bcontrabando\b", r"\bingresso il[ií]cito\b",
            r"\bobjeto proibido\b", r"\bmaterial proibido\b"
        ],
        "examples": [
            "arremesso para interior da unidade",
            "entrada de material proibido",
            "contrabando apreendido"
        ]
    },
    {
        "macroclasse": "DISCIPLINA",
        "classe": "VIOLENCIA",
        "subclasse": "AGRESSAO_BRIGA",
        "grau_risco": 9,
        "criticidade": "CRITICA",
        "patterns": [
            r"\bagress[aã]o\b", r"\bagred", r"\bbriga\b", r"\bluta corporal\b",
            r"\bvias de fato\b", r"\bespanc"
        ],
        "examples": [
            "agressao entre internos",
            "briga na cela",
            "vias de fato"
        ]
    },
    {
        "macroclasse": "DISCIPLINA",
        "classe": "VIOLENCIA",
        "subclasse": "AMEACA_COACAO",
        "grau_risco": 8,
        "criticidade": "ALTA",
        "patterns": [
            r"\bamea[cç]a\b", r"\bamea[cç]ou\b", r"\bintimid", r"\bcoa[cç][aã]o\b"
        ],
        "examples": [
            "ameaca contra servidor",
            "coacao entre internos",
            "interno intimidou outro"
        ]
    },
    {
        "macroclasse": "DISCIPLINA",
        "classe": "INDISCIPLINA",
        "subclasse": "DESOBEDIENCIA_TUMULTO",
        "grau_risco": 7,
        "criticidade": "ALTA",
        "patterns": [
            r"\bindisciplina\b", r"\bdesobedi", r"\binsubordina", r"\bdesacato\b",
            r"\btumulto\b", r"\bdesordem\b", r"\bmotim\b", r"\brebeli[aã]o\b"
        ],
        "examples": [
            "ato de indisciplina",
            "desobediencia a ordem legal",
            "tumulto no pavilhao"
        ]
    },
    {
        "macroclasse": "PATRIMONIO",
        "classe": "DANO",
        "subclasse": "DANO_AO_PATRIMONIO",
        "grau_risco": 7,
        "criticidade": "ALTA",
        "patterns": [
            r"\bdano\b", r"\bdepreda", r"\bquebra\b", r"\bdestrui", r"\binc[eê]ndio\b"
        ],
        "examples": [
            "dano ao patrimonio publico",
            "depredacao da cela",
            "quebra de estrutura"
        ]
    },
    {
        "macroclasse": "SAUDE",
        "classe": "AUTOLESAO",
        "subclasse": "AUTOLESAO_OU_SUICIDIO",
        "grau_risco": 8,
        "criticidade": "ALTA",
        "patterns": [
            r"\bautoles", r"\bautomutil", r"\bsuic[ií]dio\b", r"\btentativa de suic[ií]dio\b",
            r"\benforc", r"\bauto exterm"
        ],
        "examples": [
            "autolesao em cela",
            "tentativa de suicidio",
            "interno se automutilou"
        ]
    },
    {
        "macroclasse": "VISITA",
        "classe": "VISITA_IRREGULAR",
        "subclasse": "IRREGULARIDADE_DE_VISITA",
        "grau_risco": 6,
        "criticidade": "MEDIA",
        "patterns": [
            r"\bvisit", r"\bvisita irregular\b", r"\bvisitante\b", r"\bentrada irregular\b"
        ],
        "examples": [
            "irregularidade em visita",
            "visitante com material nao autorizado",
            "entrada irregular de visitante"
        ]
    },
    {
        "macroclasse": "SERVIDOR",
        "classe": "CONDUTA_FUNCIONAL",
        "subclasse": "SERVIDOR_ENVOLVIDO",
        "grau_risco": 8,
        "criticidade": "ALTA",
        "patterns": [
            r"\bservidor\b", r"\bpolicial penal\b", r"\bagente penitenci[aá]rio\b",
            r"\bfuncion[aá]rio\b", r"\bconduta funcional\b"
        ],
        "examples": [
            "servidor envolvido na ocorrencia",
            "conduta irregular de servidor",
            "policial penal citado"
        ]
    },
    {
        "macroclasse": "OPERACIONAL",
        "classe": "REVISTA_FISCALIZACAO",
        "subclasse": "REVISTA_OU_INSPECAO",
        "grau_risco": 4,
        "criticidade": "MEDIA",
        "patterns": [
            r"\brevista\b", r"\binspe[cç][aã]o\b", r"\bfiscaliza", r"\bvarredura\b", r"\bbusca\b"
        ],
        "examples": [
            "revista de cela",
            "inspecao de rotina",
            "busca em pavilhao"
        ]
    },
    {
        "macroclasse": "OPERACIONAL",
        "classe": "PROCEDIMENTO_ADMINISTRATIVO",
        "subclasse": "REGISTRO_OPERACIONAL",
        "grau_risco": 2,
        "criticidade": "BAIXA",
        "patterns": [
            r"\bprocedimento\b", r"\bregistro\b", r"\bcomunica[cç][aã]o\b",
            r"\bapoio\b", r"\bacompanhamento\b", r"\borienta[cç][aã]o\b"
        ],
        "examples": [
            "registro operacional",
            "procedimento administrativo",
            "apoio a atividade"
        ]
    },
    {
        "macroclasse": "OUTROS",
        "classe": "OUTROS",
        "subclasse": "NAO_CLASSIFICADO",
        "grau_risco": 1,
        "criticidade": "BAIXA",
        "patterns": [],
        "examples": [
            "outros",
            "nao classificado",
            "diversos"
        ]
    }
]


# ============================================================
# NORMALIZACAO
# ============================================================

MAPA_SUBSTITUICAO = {
    "cel.": "celular",
    "cel ": "celular ",
    "tel ": "telefone ",
    "apreensao": "apreensao",
    "entorpec.": "entorpecente",
    "entorpec ": "entorpecente ",
    "subst ": "substancia ",
    "obj ": "objeto ",
    "perfuro cortante": "perfurocortante",
    "pol penal": "policial penal",
    "ag penitenciario": "agente penitenciario",
    "ag penitenciária": "agente penitenciario",
    "ag penit": "agente penitenciario",
    "evasao": "evasao",
    "rebelião": "rebelião",
    "desob.": "desobediencia",
    "auto lesao": "autolesao",
    "auto-exterm": "auto exterm",
}

def remover_acentos(txt):
    if txt is None:
        return ""
    return "".join(
        c for c in unicodedata.normalize("NFKD", str(txt))
        if not unicodedata.combining(c)
    )

def normalizar_texto(txt):
    txt = remover_acentos(txt).lower().strip()
    txt = re.sub(r"[\r\n\t]+", " ", txt)
    txt = re.sub(r"[/_]+", " ", txt)
    txt = re.sub(r"[^a-z0-9\s\-]", " ", txt)
    txt = re.sub(r"\s+", " ", txt).strip()

    for k, v in MAPA_SUBSTITUICAO.items():
        txt = txt.replace(k, v)

    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

def gerar_id_texto(motivo, registro):
    base = f"{motivo or ''}|{registro or ''}"
    return hashlib.md5(base.encode("utf-8")).hexdigest()

def criticidade_num(txt):
    mapa = {"BAIXA": 1, "MEDIA": 2, "ALTA": 3, "CRITICA": 4}
    return mapa.get(txt, 0)


# ============================================================
# CONSTRUIR BASE DE REFERENCIA SEMANTICA
# ============================================================

refs = []
for item in TAXONOMIA:
    for ex in item["examples"]:
        refs.append({
            "macroclasse": item["macroclasse"],
            "classe": item["classe"],
            "subclasse": item["subclasse"],
            "grau_risco": item["grau_risco"],
            "criticidade": item["criticidade"],
            "texto_referencia": normalizar_texto(ex),
            "patterns": item["patterns"]
        })

if SKLEARN_OK:
    corpus_ref = [r["texto_referencia"] for r in refs]
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)
    ref_matrix = vectorizer.fit_transform(corpus_ref)
else:
    vectorizer = None
    ref_matrix = None


# ============================================================
# CLASSIFICADOR HIBRIDO
# ============================================================

def classificar_por_regra(texto_norm):
    melhor = None
    melhor_score = -1
    sinais = []

    for item in TAXONOMIA:
        if item["classe"] == "OUTROS":
            continue

        hits = []
        for p in item["patterns"]:
            if re.search(p, texto_norm):
                hits.append(p)

        score = len(hits)

        if score > 0:
            score_ajustado = score * 10 + item["grau_risco"] + criticidade_num(item["criticidade"])
            if score_ajustado > melhor_score:
                melhor_score = score_ajustado
                melhor = item
                sinais = hits

    if melhor is None:
        return None

    confianca = min(99, 70 + len(sinais) * 8 + melhor["grau_risco"])
    return {
        "macroclasse": melhor["macroclasse"],
        "classe": melhor["classe"],
        "subclasse": melhor["subclasse"],
        "grau_risco": melhor["grau_risco"],
        "criticidade": melhor["criticidade"],
        "confianca_classificacao": float(confianca),
        "origem_classificacao": "REGRA",
        "justificativa_classificacao": f"match_regra:{', '.join(sinais[:8])}",
        "sinais_detectados": json.dumps(sinais[:20], ensure_ascii=False)
    }

def classificar_por_similaridade(texto_norm):
    if not texto_norm:
        return None

    if SKLEARN_OK:
        vec = vectorizer.transform([texto_norm])
        sims = cosine_similarity(vec, ref_matrix)[0]
        idx = int(sims.argmax())
        score = float(sims[idx])
        ref = refs[idx]

        if score < 0.33:
            return None

        confianca = round(score * 100, 2)
        return {
            "macroclasse": ref["macroclasse"],
            "classe": ref["classe"],
            "subclasse": ref["subclasse"],
            "grau_risco": ref["grau_risco"],
            "criticidade": ref["criticidade"],
            "confianca_classificacao": confianca,
            "origem_classificacao": "SIMILARIDADE",
            "justificativa_classificacao": f"match_semantico:{ref['texto_referencia']}",
            "sinais_detectados": json.dumps([ref["texto_referencia"]], ensure_ascii=False)
        }

    melhor = None
    melhor_score = -1.0

    for ref in refs:
        score = SequenceMatcher(None, texto_norm, ref["texto_referencia"]).ratio()
        if score > melhor_score:
            melhor_score = score
            melhor = ref

    if melhor is None or melhor_score < 0.45:
        return None

    return {
        "macroclasse": melhor["macroclasse"],
        "classe": melhor["classe"],
        "subclasse": melhor["subclasse"],
        "grau_risco": melhor["grau_risco"],
        "criticidade": melhor["criticidade"],
        "confianca_classificacao": round(melhor_score * 100, 2),
        "origem_classificacao": "SIMILARIDADE",
        "justificativa_classificacao": f"match_aproximado:{melhor['texto_referencia']}",
        "sinais_detectados": json.dumps([melhor["texto_referencia"]], ensure_ascii=False)
    }

def classificar_texto(motivo_original, registro_original):
    motivo_original = motivo_original or ""
    registro_original = registro_original or ""

    motivo_norm = normalizar_texto(motivo_original)
    registro_norm = normalizar_texto(registro_original)
    texto_norm = normalizar_texto(f"{motivo_original} | {registro_original}")

    # prioridade alta para regra
    r = classificar_por_regra(texto_norm)
    if r is not None:
        return {
            "motivo_normalizado": motivo_norm,
            "registro_normalizado": registro_norm,
            "texto_classificacao_normalizado": texto_norm,
            **r
        }

    # fallback semantico
    s = classificar_por_similaridade(texto_norm)
    if s is not None:
        return {
            "motivo_normalizado": motivo_norm,
            "registro_normalizado": registro_norm,
            "texto_classificacao_normalizado": texto_norm,
            **s
        }

    return {
        "motivo_normalizado": motivo_norm,
        "registro_normalizado": registro_norm,
        "texto_classificacao_normalizado": texto_norm,
        "macroclasse": "OUTROS",
        "classe": "OUTROS",
        "subclasse": "NAO_CLASSIFICADO",
        "grau_risco": 1,
        "criticidade": "BAIXA",
        "confianca_classificacao": 15.0,
        "origem_classificacao": "FALLBACK",
        "justificativa_classificacao": "sem_match_regra_ou_semantico",
        "sinais_detectados": json.dumps([], ensure_ascii=False)
    }


# ============================================================
# BASE DE TEXTOS DISTINTOS PARA CLASSIFICAR
# ============================================================

df_texto_base = spark.sql("""
    select distinct
        md5(concat_ws('|', coalesce(motivo, ''), coalesce(registro, ''))) as id_texto_classificacao,
        coalesce(motivo, '') as motivo_original,
        coalesce(registro, '') as registro_original
    from gold.sinp_fat_ocorrencia_livro
""")

tabela = "tmp_ocorrencia_livro_texto_base"

df_texto_base.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_texto_base, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_ocorrencia_livro_texto_base")


# ============================================================
# IDENTIFICAR SOMENTE TEXTOS NOVOS
# ============================================================

tem_dim_anterior = spark.sql("show tables in gold like 'sinp_dim_classificacao_ocorrencia_livro'").count() > 0

if tem_dim_anterior:
    spark.sql("refresh table gold.sinp_dim_classificacao_ocorrencia_livro")

    df_texto_novo = spark.sql("""
        select
            b.id_texto_classificacao,
            b.motivo_original,
            b.registro_original
        from gold.tmp_ocorrencia_livro_texto_base b
        left join gold.sinp_dim_classificacao_ocorrencia_livro d
            on b.id_texto_classificacao = d.id_texto_classificacao
        where d.id_texto_classificacao is null
    """)
else:
    df_texto_novo = spark.sql("""
        select
            id_texto_classificacao,
            motivo_original,
            registro_original
        from gold.tmp_ocorrencia_livro_texto_base
    """)

tabela = "tmp_ocorrencia_livro_texto_novo"

df_texto_novo.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_texto_novo, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_ocorrencia_livro_texto_novo")


# ============================================================
# CLASSIFICAR EM PYTHON APENAS OS NOVOS
# ============================================================

schema_classificacao = T.StructType([
    T.StructField("id_texto_classificacao", T.StringType(), False),
    T.StructField("motivo_original", T.StringType(), True),
    T.StructField("registro_original", T.StringType(), True),
    T.StructField("motivo_normalizado", T.StringType(), True),
    T.StructField("registro_normalizado", T.StringType(), True),
    T.StructField("texto_classificacao_normalizado", T.StringType(), True),
    T.StructField("macroclasse", T.StringType(), True),
    T.StructField("classe", T.StringType(), True),
    T.StructField("subclasse", T.StringType(), True),
    T.StructField("grau_risco", T.IntegerType(), True),
    T.StructField("criticidade", T.StringType(), True),
    T.StructField("confianca_classificacao", T.DoubleType(), True),
    T.StructField("origem_classificacao", T.StringType(), True),
    T.StructField("versao_modelo", T.StringType(), True),
    T.StructField("justificativa_classificacao", T.StringType(), True),
    T.StructField("sinais_detectados", T.StringType(), True),
    T.StructField("dt_classificacao", T.TimestampType(), True),
])

novos = df_texto_novo.collect()

registros_classificados = []
versao_modelo = "CLASSIF_LIVRO_V1"

for row in novos:
    motivo_original = row["motivo_original"]
    registro_original = row["registro_original"]

    c = classificar_texto(motivo_original, registro_original)

    registros_classificados.append({
        "id_texto_classificacao": row["id_texto_classificacao"],
        "motivo_original": motivo_original,
        "registro_original": registro_original,
        "motivo_normalizado": c["motivo_normalizado"],
        "registro_normalizado": c["registro_normalizado"],
        "texto_classificacao_normalizado": c["texto_classificacao_normalizado"],
        "macroclasse": c["macroclasse"],
        "classe": c["classe"],
        "subclasse": c["subclasse"],
        "grau_risco": int(c["grau_risco"]),
        "criticidade": c["criticidade"],
        "confianca_classificacao": float(c["confianca_classificacao"]),
        "origem_classificacao": c["origem_classificacao"],
        "versao_modelo": versao_modelo,
        "justificativa_classificacao": c["justificativa_classificacao"],
        "sinais_detectados": c["sinais_detectados"],
        "dt_classificacao": datetime.now()
    })

if len(registros_classificados) > 0:
    df_classificacao_nova = spark.createDataFrame(pd.DataFrame(registros_classificados), schema=schema_classificacao)
else:
    df_classificacao_nova = spark.createDataFrame([], schema=schema_classificacao)

tabela = "tmp_dim_classificacao_ocorrencia_livro_novo"

df_classificacao_nova.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_classificacao_nova, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_dim_classificacao_ocorrencia_livro_novo")


# ============================================================
# DIM FINAL DE CLASSIFICACAO
# ============================================================

if tem_dim_anterior:
    df_dim_final = spark.sql("""
        select * from gold.sinp_dim_classificacao_ocorrencia_livro
        union all
        select * from gold.tmp_dim_classificacao_ocorrencia_livro_novo
    """)
else:
    df_dim_final = spark.sql("""
        select * from gold.tmp_dim_classificacao_ocorrencia_livro_novo
    """)

tabela = "tmp_dim_classificacao_ocorrencia_livro_final"

df_dim_final.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_dim_final, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_dim_classificacao_ocorrencia_livro_final")


df_dim_persistida = spark.sql("""
    select distinct
        id_texto_classificacao,
        motivo_original,
        registro_original,
        motivo_normalizado,
        registro_normalizado,
        texto_classificacao_normalizado,
        macroclasse,
        classe,
        subclasse,
        grau_risco,
        criticidade,
        confianca_classificacao,
        origem_classificacao,
        versao_modelo,
        justificativa_classificacao,
        sinais_detectados,
        dt_classificacao
    from gold.tmp_dim_classificacao_ocorrencia_livro_final
""")

tabela = "sinp_dim_classificacao_ocorrencia_livro"

df_dim_persistida.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_dim_persistida, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_texto_classificacao")


# ============================================================
# ENRIQUECER FATO DO LIVRO
# ============================================================

df_fat_ocorrencia_livro_classificada = spark.sql("""
    select
        f.*,

        md5(concat_ws('|', coalesce(f.motivo, ''), coalesce(f.registro, ''))) as id_texto_classificacao,

        d.motivo_normalizado,
        d.registro_normalizado,
        d.texto_classificacao_normalizado,

        d.macroclasse as macroclasse_motivo,
        d.classe as classe_motivo,
        d.subclasse as subclasse_motivo,
        d.grau_risco as grau_risco_motivo,
        d.criticidade as criticidade_motivo,
        d.confianca_classificacao,
        d.origem_classificacao,
        d.versao_modelo,
        d.justificativa_classificacao,
        d.sinais_detectados,
        d.dt_classificacao,

        case
            when d.grau_risco >= 8 then 1
            else 0
        end as flag_motivo_critico,

        case
            when d.classe in ('APREENSAO_ILICITO', 'CONTRABANDO') then 1
            else 0
        end as flag_motivo_ilicito,

        case
            when d.classe in ('VIOLENCIA') then 1
            else 0
        end as flag_motivo_violencia,

        case
            when d.classe in ('FUGA_EVASAO') then 1
            else 0
        end as flag_motivo_fuga,

        case
            when d.classe in ('VISITA_IRREGULAR') then 1
            else 0
        end as flag_motivo_visita,

        case
            when d.classe in ('CONDUTA_FUNCIONAL') then 1
            else 0
        end as flag_motivo_servidor,

        coalesce(f.score_complexidade_basica, 0) + coalesce(d.grau_risco, 0) as score_risco_ocorrencia_livro
    from gold.sinp_fat_ocorrencia_livro f
    left join gold.sinp_dim_classificacao_ocorrencia_livro d
        on md5(concat_ws('|', coalesce(f.motivo, ''), coalesce(f.registro, ''))) = d.id_texto_classificacao
""")

tabela = "sinp_fat_ocorrencia_livro_classificada"

df_fat_ocorrencia_livro_classificada.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_ocorrencia_livro_classificada, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_fato_ocorrencia")


# ============================================================
# VALIDACOES
# ============================================================

spark.sql("""
select
    count(*) as total_ocorrencias,
    sum(case when macroclasse_motivo is not null then 1 else 0 end) as ocorrencias_classificadas,
    sum(case when flag_motivo_critico = 1 then 1 else 0 end) as ocorrencias_criticas,
    sum(case when flag_motivo_ilicito = 1 then 1 else 0 end) as ocorrencias_ilicitas,
    sum(case when flag_motivo_violencia = 1 then 1 else 0 end) as ocorrencias_violentas,
    sum(case when flag_motivo_fuga = 1 then 1 else 0 end) as ocorrencias_fuga
from gold.sinp_fat_ocorrencia_livro_classificada
""").show(truncate=False)

spark.sql("""
select
    macroclasse,
    classe,
    subclasse,
    count(*) as qtd
from gold.sinp_dim_classificacao_ocorrencia_livro
group by
    macroclasse,
    classe,
    subclasse
order by qtd desc, macroclasse, classe, subclasse
""").show(200, truncate=False)

spark.sql("""
select
    origem_classificacao,
    count(*) as qtd
from gold.sinp_dim_classificacao_ocorrencia_livro
group by origem_classificacao
order by qtd desc
""").show(truncate=False)

spark.sql("""
select
    id_fato_ocorrencia,
    motivo,
    registro,
    macroclasse_motivo,
    classe_motivo,
    subclasse_motivo,
    grau_risco_motivo,
    confianca_classificacao,
    origem_classificacao,
    score_risco_ocorrencia_livro
from gold.sinp_fat_ocorrencia_livro_classificada
order by score_risco_ocorrencia_livro desc, confianca_classificacao desc
""").show(100, truncate=False)

NameError: name 'spark' is not defined